# 10 · Fluxo LangGraph com Pydantic: do agente livre ao caminho auditável

No notebook 09, o agente **ReAct** decide sozinho a ordem das ferramentas. Isso é flexível,
mas mostrou um problema real: o LLM convertia "15 por 9,5" em 155/95 em 2 de 3 tentativas.
Este notebook aplica duas ideias dos exemplos da disciplina (Colabs de Pydantic e de
LangGraph, e o repositório *mercado-simples-langgraph*):

| Ideia | Onde está |
|:--|:--|
| **Pydantic** como contrato dos dados: normaliza, rejeita valores impossíveis e devolve o erro ao LLM para correção | `src/agente/esquemas.py` |
| **LangGraph** com caminho explícito: o LLM só extrai e redige, e validar e calcular são código | `src/agente/fluxo.py` |
| **Modo demonstração** sem chave: as mesmas etapas, com regras no lugar do LLM | `src/agente/fluxo.py` |
| **Especificação → testes**: cada critério tem um ID citado nos testes | `specs/01-triagem.md` |

Ao final, um experimento compara o agente ReAct com o fluxo LangGraph na mesma tarefa.

In [1]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import json
import os
import time

import pandas as pd
from pydantic import ValidationError

from src.agente.agente import conversar, criar_agente, criar_modelo_llm, provedor
from src.agente.esquemas import Paciente, PacienteParcial, traduzir_erros, validar
from src.agente.fluxo import construir_fluxo, extrair_por_regras, triar
from src.utils.io import salvar_tabela

SECAO = "10-fluxo-langgraph"
PROVEDOR = provedor()
print(f"Provedor de LLM: {PROVEDOR}")

Provedor de LLM: openai


## 1. Pydantic: o contrato dos dados

O esquema `Paciente` aceita dados em formato humano e os normaliza antes de qualquer
cálculo, como o validador de preço "R$ 19,90" do Colab de Pydantic.

In [2]:
paciente = Paciente.model_validate({
    "idade": "58 anos", "sexo": "homem", "altura_cm": "1,72 m", "peso_kg": "94 kg",
    "pressao_sistolica": 15, "pressao_diastolica": "9,5",
    "colesterol": "acima do normal", "glicose": "normais",
    "fumante": True, "consome_alcool": False, "fisicamente_ativo": False,
})
print(paciente.model_dump())
print(f"IMC: {paciente.imc:.1f}")

{'idade': 58.0, 'sexo': 'masculino', 'altura_cm': 172.0, 'peso_kg': 94.0, 'pressao_sistolica': 150, 'pressao_diastolica': 95, 'colesterol': 2, 'glicose': 1, 'fumante': True, 'consome_alcool': False, 'fisicamente_ativo': False}
IMC: 31.8


Valores impossíveis ou incoerentes são **rejeitados** antes de chegar à rede, e as mensagens
de erro são traduzidas para português. Valores plausíveis, mas fora da faixa do treino,
geram apenas um **aviso**.

In [3]:
for alteracao in [{"pressao_diastolica": 160}, {"idade": 5, "altura_cm": 300}, {"exame_extra": "ECG"}]:
    try:
        Paciente.model_validate({**paciente.model_dump(), **alteracao})
    except ValidationError as erro:
        print(alteracao, "→", traduzir_erros(erro))

print("\nAviso de extrapolação:", Paciente.model_validate({**paciente.model_dump(), "idade": 78}).avisos())

{'pressao_diastolica': 160} → ['A pressão diastólica precisa ser menor que a sistólica.']
{'idade': 5, 'altura_cm': 300} → ['idade: valor abaixo do mínimo plausível (18.0).', 'altura: valor acima do máximo plausível (230.0).']
{'exame_extra': 'ECG'} → ['dados: valor não reconhecido.']

Aviso de extrapolação: ['idade = 78 está fora da faixa vista no treino (30–65); a previsão é uma extrapolação e merece menos confiança.']


Quando o esquema é a assinatura de uma ferramenta, os limites viram o **JSON Schema** que o
LLM lê. É o mesmo esquema que valida a resposta dele:

In [4]:
esquema = Paciente.model_json_schema()["properties"]
pd.DataFrame(esquema).T[["type", "minimum", "maximum", "enum", "description"]].fillna("")

,type,minimum,maximum,enum,description
idade,number,18,110,,Idade em anos
sexo,string,,,"[feminino, masculino]",
altura_cm,number,100,230,,"Altura em centímetros (1,72 m = 172)"
peso_kg,number,30,250,,Peso em quilogramas
pressao_sistolica,integer,60,260,,Pressão sistólica em mmHg (150)
pressao_diastolica,integer,30,200,,Pressão diastólica em mmHg (95)
colesterol,integer,,,"[1, 2, 3]","1 normal, 2 acima do normal, 3 muito acima"
glicose,integer,,,"[1, 2, 3]","1 normal, 2 acima do normal, 3 muito acima"
fumante,boolean,,,,
consome_alcool,boolean,,,,


## 2. O grafo

Cada nó é uma função comum que devolve os campos do estado que quer atualizar, e as arestas
condicionais decidem o próximo passo. O GitHub desenha este diagrama a partir do código Mermaid:

In [5]:
fluxo_demo = construir_fluxo()  # sem LLM: modo demonstração
print(fluxo_demo.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	interpretar(interpretar)
	validar(validar)
	perguntar(perguntar)
	corrigir(corrigir)
	avaliar(avaliar)
	explicar(explicar)
	responder_pergunta(responder_pergunta)
	__end__([<p>__end__</p>]):::last
	__start__ --> interpretar;
	avaliar --> explicar;
	interpretar -. &nbsp;pergunta&nbsp; .-> responder_pergunta;
	interpretar -. &nbsp;triagem&nbsp; .-> validar;
	validar -.-> avaliar;
	validar -.-> corrigir;
	validar -.-> perguntar;
	corrigir --> __end__;
	explicar --> __end__;
	perguntar --> __end__;
	responder_pergunta --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



```mermaid
flowchart TD
    S([início]) --> I[interpretar<br/>LLM ou regras]
    I -->|pergunta| Q[responder_pergunta]
    I -->|triagem| V[validar<br/>Pydantic]
    V -->|faltam dados| P[perguntar]
    V -->|valor inválido| C[corrigir]
    V -->|completo| A[avaliar<br/>rede neural]
    A --> E[explicar<br/>LLM ou modelo de texto]
    Q --> F([fim]); P --> F; C --> F; E --> F
```

**Quem faz o quê:** o LLM (ou as regras) só atua em *interpretar*, *explicar* e
*responder_pergunta*. Validar e avaliar são sempre código, determinísticos e testáveis.

## 3. Modo demonstração: o grafo inteiro sem IA

Sem chave de API, *interpretar* usa expressões regulares e *explicar* usa um modelo de texto.
A validação e a rede neural são as mesmas.

In [6]:
def mostrar(r):
    for passo in r["rastreio"]:
        print("  →", passo)
    print(r["resposta"], "\n")

r = triar("Mulher, 61 anos, pressão 14 por 9. Devo encaminhar?", fluxo=fluxo_demo)
mostrar(r)
r = triar("1,60 m, 70 kg, colesterol e glicose normais, não fuma, não bebe, faz caminhada", r["parcial"], fluxo=fluxo_demo)
mostrar(r)
r = triar("e se a pressão fosse 12 por 8?", r["parcial"], fluxo=fluxo_demo)
mostrar(r)

  → interpretar (regras) → triagem; campos nesta mensagem: idade, sexo, pressao_texto
  → validar (Pydantic) → faltando: altura, peso, colesterol (normal, acima do normal ou muito acima), glicose (normal, acima do normal ou muito acima), se fuma, se consome álcool, se pratica atividade física
  → perguntar (modelo de texto) → dados faltando
_[Modo demonstração — sem IA: dados extraídos por regras e resposta montada por modelo de texto.]_

Para calcular o risco, ainda preciso de:
- altura
- peso
- colesterol (normal, acima do normal ou muito acima)
- glicose (normal, acima do normal ou muito acima)
- se fuma
- se consome álcool
- se pratica atividade física 



  → interpretar (regras) → triagem; campos nesta mensagem: altura_cm, peso_kg, colesterol, glicose, fumante, consome_alcool, fisicamente_ativo
  → validar (Pydantic) → completo
  → avaliar (rede neural) → P = 80.5%, encaminhar para investigação
  → explicar (modelo de texto)
_[Modo demonstração — sem IA: dados extraídos por regras e resposta montada por modelo de texto.]_

**Dados usados:** 61 anos, feminino, 160 cm, 70 kg, pressão 140/90, colesterol normal, glicose normal, fuma: não, álcool: não, fisicamente ativo: sim.

**Probabilidade:** 80% → **encaminhar para investigação** (limiar de triagem 38%).

**Fatores que mais pesaram:** pressão sistólica (140): +28,4 p.p. (aumenta o risco); idade (61): +1,8 p.p. (aumenta o risco)

A decisão final é da equipe clínica. 

  → interpretar (regras) → triagem; campos nesta mensagem: pressao_texto
  → validar (Pydantic) → completo
  → avaliar (rede neural) → P = 49.9%, encaminhar para investigação
  → explicar (modelo de texto)
_[Modo demonstraç

Os dados **se acumulam** entre mensagens: o fluxo perguntou o que faltava, calculou quando
recebeu e, no "e se…", mudou só a pressão e recalculou.

## 4. Experimento: agente ReAct × fluxo LangGraph

A mesma mensagem, 3 vezes em cada abordagem, com o mesmo LLM. Medimos se a pressão foi
extraída corretamente (150/95), quantas chamadas ao LLM foram feitas e o tempo. Requer chave
de API; sem ela, a seção é pulada.

In [7]:
MENSAGEM = ("Homem de 58 anos, 1,72 m e 94 kg, pressão 15 por 9,5, colesterol acima do normal, "
            "glicose normal, fuma, não bebe e é sedentário. Qual o risco?")
REPETICOES = 3

if PROVEDOR == "demo":
    print("Pulado: sem chave de API (defina OPENAI_API_KEY ou DEEPSEEK_API_KEY no .env).")
else:
    llm = criar_modelo_llm()
    agente, fluxo_llm = criar_agente(llm), construir_fluxo(llm)
    linhas = []
    for i in range(REPETICOES):
        inicio = time.perf_counter()
        resposta, chamadas = conversar(agente, [{"role": "user", "content": MENSAGEM}])
        args = next((c["argumentos"] for c in chamadas if c["ferramenta"] == "avaliar_paciente"), {})
        linhas.append({"abordagem": "Agente ReAct (notebook 09)", "rodada": i + 1,
                       "pressao": f"{args.get('pressao_sistolica')}/{args.get('pressao_diastolica')}",
                       "chamadas_ao_llm": len(chamadas) + 1, "segundos": time.perf_counter() - inicio})

        inicio = time.perf_counter()
        r = triar(MENSAGEM, fluxo=fluxo_llm)
        p = r["parcial"]
        pressao = validar(PacienteParcial.model_validate(p))[0]
        linhas.append({"abordagem": "Fluxo LangGraph", "rodada": i + 1,
                       "pressao": f"{pressao.pressao_sistolica}/{pressao.pressao_diastolica}" if pressao else "—",
                       "chamadas_ao_llm": 2, "segundos": time.perf_counter() - inicio})
    experimento = pd.DataFrame(linhas)
    experimento["pressao_correta"] = experimento["pressao"] == "150/95"
    salvar_tabela(experimento.round(2), SECAO, "react-vs-langgraph")
    resumo = experimento.groupby("abordagem").agg(
        acertos_pressao=("pressao_correta", "sum"), chamadas_ao_llm=("chamadas_ao_llm", "mean"),
        segundos=("segundos", "mean"),
    ).round(1)
    salvar_tabela(resumo, SECAO, "react-vs-langgraph-resumo")
    display(experimento.round(2))
    display(resumo)

,abordagem,rodada,pressao,chamadas_ao_llm,segundos,pressao_correta
0,Agente ReAct (notebook 09),1,150/95,3,4.67,True
1,Fluxo LangGraph,1,150/95,2,15.92,True
2,Agente ReAct (notebook 09),2,150/95,3,4.16,True
3,Fluxo LangGraph,2,150/95,2,3.38,True
4,Agente ReAct (notebook 09),3,150/95,3,4.38,True
5,Fluxo LangGraph,3,150/95,2,5.89,True


,acertos_pressao,chamadas_ao_llm,segundos
abordagem,,,
Agente ReAct (notebook 09),3,3.0,4.4
Fluxo LangGraph,3,2.0,8.4


## Conclusões

**O experimento.** As duas abordagens extraíram a pressão corretamente nas 3 rodadas. O agente
ReAct só acerta hoje porque, depois do erro do "155", ganhou a ferramenta `converter_pressao`
(notebook 09). O fluxo LangGraph fez menos chamadas ao LLM (2 contra 3: extrair e explicar).
No tempo, as duas ficaram parecidas, salvo a primeira rodada do fluxo, mais lenta pelo
aquecimento da conexão. Três rodadas são pouco para comparar velocidade; o ponto principal é
outro.

**A diferença está nas garantias, não na média.**

| | Agente ReAct | Fluxo LangGraph |
|:--|:--|:--|
| Quem decide a ordem dos passos | o LLM | o código (arestas do grafo) |
| Conversão da pressão | depende de o LLM chamar a ferramenta | sempre em código: não há como pular |
| Dados faltando | depende da instrução ser seguida | o nó `validar` impede a chamada à rede |
| Saída inválida do LLM | o erro volta pela ferramenta | Pydantic rejeita e pede correção (até 2 vezes) |
| Sem chave de API | não funciona | modo demonstração com as mesmas etapas |
| Auditoria | lista de chamadas de ferramentas | rastreio de cada nó |
| Flexibilidade para pedidos inesperados | maior | menor: só os caminhos desenhados |

Para **triagem em saúde**, previsibilidade vale mais que flexibilidade: o fluxo LangGraph é a
escolha da aplicação (aba Assistente). O agente ReAct continua no notebook 09 como
referência do padrão mais livre.

**O que veio dos exemplos da disciplina:**

* do Colab de Pydantic: o esquema que normaliza formatos humanos, rejeita valores impossíveis
  e devolve o erro ao LLM para correção;
* do Colab de LangGraph: o `StateGraph` com estado compartilhado, nós como funções e um
  `rastreio` acumulado com `operator.add`;
* do repositório *mercado-simples-langgraph*: o modo demonstração sem chave, a regra "não usar
  IA para o que o código decide", a especificação com IDs citados nos testes e o GitHub
  Actions.